# Inhibitory neurons analysis

Here we analyze presynaptic partner's properties of inhibitory neurons.
In addition to inhibitory neuron's inputs, we also analyze excitatory neuron's inputs for comparison.

# Setup

In [1]:
from caveclient import CAVEclient
import numpy as np
import pandas as pd
from pathlib import Path
import glob
import os
import microns_datacleaner as mic
import microns_datacleaner.filters as fl
from scipy.ndimage import gaussian_filter
import matplotlib.cm as cm
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import TwoSlopeNorm

In [2]:
# open dataset and set version
client = CAVEclient("minnie65_public")
client.materialize.version=1300

In [3]:
# list all the tables
client.materialize.get_tables()

['baylor_gnn_cell_type_fine_model_v2',
 'nucleus_alternative_points',
 'allen_column_mtypes_v2',
 'bodor_pt_cells',
 'aibs_metamodel_mtypes_v661_v2',
 'allen_v1_column_types_slanted_ref',
 'aibs_column_nonneuronal_ref',
 'nucleus_ref_neuron_svm',
 'apl_functional_coreg_vess_fwd',
 'vortex_compartment_targets',
 'baylor_log_reg_cell_type_coarse_v1',
 'functional_properties_v3_bcm',
 'gamlin_2023_mcs',
 'l5et_column',
 'pt_synapse_targets',
 'coregistration_manual_v4',
 'cg_cell_type_calls',
 'synapses_pni_2',
 'nucleus_detection_v0',
 'vortex_manual_nodes_of_ranvier',
 'vortex_astrocyte_proofreading_status',
 'bodor_pt_target_proofread',
 'nucleus_functional_area_assignment',
 'coregistration_auto_phase3_fwd_apl_vess_combined_v2',
 'synapse_target_structure',
 'coregistration_auto_phase3_fwd_v2',
 'gamlin_2023_mcs_met_types',
 'vortex_manual_myelination_v0',
 'proofreading_status_and_strategy',
 'synapse_target_predictions_ssa',
 'aibs_metamodel_celltypes_v661']

In [4]:
# Query tables

# Get list of neurons with extended proofreading of dendrites
dend_extended_df = client.materialize.query_table('proofreading_status_and_strategy',filter_in_dict={"strategy_dendrite": ["dendrite_extended"]})

# Coregistration table:   Between EM and 2P functional image
coregistration_manual_v4 = client.materialize.query_table('coregistration_manual_v4')

# nucleus_detection_v0: Table of each nucleus and their ID
nucleus_detection_v0= client.materialize.query_table('nucleus_detection_v0')
#nucleus_detection_v0.head()

# aibs_metamodel_celltypes_v661:    Cell type and layer
aibs_metamodel_celltypes_v661= client.materialize.query_table('aibs_metamodel_celltypes_v661')
#aibs_metamodel_celltypes_v661.head()

# neucleus_functional_area_assignment:      area [V1, LM, AL, RL]
nucleus_functional_area_assignment= client.materialize.query_table('nucleus_functional_area_assignment')
nucleus_functional_area_assignment.head()

# functional_properties_v3_bcm
functional_properties_v3_bcm = client.materialize.tables.functional_properties_v3_bcm().query()
# mask = functional_properties_v3_bcm['target_id'] == target_id_coreg
# functional_properties_v3_bcm[mask]

In [5]:
# Set path
path_project = "/media/DATA1/CK/python_script/MicronsBinder_04/"

# Inhibitory neurons

Bring neuron from microns datacleaner

In [71]:
# Read from datacleaner
cleaner = mic.MicronsDataCleaner(datadir = "data", version=1300)   #Target version and download folder
cleaner.download_nucleus_data()    #Download the data
units, segments = cleaner.process_nucleus_data() #Process the downloaded data and segment into layers

Transform positions: 100%|███████████████████████████████████████████████████████████████████| 94014/94014 [00:00<00:00, 153089.54it/s]


In [95]:
# Filter inhibitory neurons
units_BC = units[units["cell_type"] == "BC"][["pt_root_id"]]
units_BPC = units[units["cell_type"] == "BPC"][["pt_root_id"]]
units_MC = units[units["cell_type"] == "MC"][["pt_root_id"]]
units_NGC = units[units["cell_type"] == "NGC"][["pt_root_id"]]

# Excitatory neurons

Bring neuron from microns datacleaner

In [144]:
# From datacleaner, bring excitatory neurons from 23 layer in V1
units_23P = fl.filter_neurons(units, cell_type=["23P"], brain_area=["V1"])
len(units_23P)

13305

# Input synapse analysis

## Inhibitory: Save functional df table

In [11]:
# === Read presynapse of inhibitory neurons and save functional dataframe of each neuron types ====
units = units_NGC      # Change over types of neuron you want
fname_df = "NGC" # When saving concatenated file

# Read functional table to see the functional inputs of inhibitory neurons
func_cols = ['pt_root_id', 'pref_ori', 'pref_dir', 'gOSI', 'gDSI', 'cc_abs']  # Only read columns we're interested in
func_props = (functional_properties_v3_bcm[func_cols].drop_duplicates(subset='pt_root_id'))

# Path
path_inhibitory = Path(path_project) / "data/data_raw_inhibitory/synapse/"
path_save = Path(path_project) / "results_inhibitory"

# Loop over neurons and build one big dataframe for each types of inhibitory neurons
all_synapse_func = []

for neuron in units['pt_root_id']:
    fname = f"{neuron}_pre_synapse_df.pkl"
    fpath = path_inhibitory / fname
    pre_synapse_df = pd.read_pickle(fpath)

    # Merge functional props onto each synapse with "pre_pt_root_id"
    pre_synapse_with_func = (pre_synapse_df.merge(
        func_props,
        left_on = "pre_pt_root_id",  # presynaptic neuron ID in synapse table
        right_on = "pt_root_id",     # same neuron ID in functional table
        how = "left")
    .drop(columns=["pt_root_id"])) # drop duplicate ID col from functional table

    # keep only synapse id + functional fields (and whatever else you want)
    synapse_func_df = pre_synapse_with_func[
        ['id', 'pre_pt_root_id', 'pref_ori', 'pref_dir', 'gOSI', 'gDSI', 'cc_abs']
    ].copy()

    # tag which postsyn neuron thse synapases belong to
    synapse_func_df['post_pt_root_id'] = neuron
    all_synapse_func.append(synapse_func_df)

# Concatenate and save them as a pkl file
synapse_func_all = pd.concat(all_synapse_func, ignore_index=True)
out_path = path_save / f"synapse_func_all_{fname_df}.pkl"
synapse_func_all.to_pickle(out_path)
print(f"Saved {len(synapse_func_all)} synapses to {out_path}")

KeyboardInterrupt: 

## Inhibitory: Read functional df table

In [6]:
path_read = Path(path_project) / "results_inhibitory"

synapse_func_BC = pd.read_pickle(path_read / "synapse_func_all_BC.pkl")
synapse_func_BPC = pd.read_pickle(path_read / "synapse_func_all_BPC.pkl")
synapse_func_MC = pd.read_pickle(path_read / "synapse_func_all_MC.pkl")
synapse_func_NGC = pd.read_pickle(path_read / "synapse_func_all_NGC.pkl")

In [7]:
# Data Cleaning  - delete rows with no functional properties and delete neurons with only single connection

# Step 1) drop rows with missing functional props (pref_ori acts as the gate)
def clean_dropna_and_drop_singletons(df, group_col="post_pt_root_id", gate_col="pref_ori"):
    df_clean = df.loc[df[gate_col].notna()].copy()

    # Step 2) drop post neurons that have only a single remaining (non-NA) synapse row
    n_per_post = df_clean.groupby(group_col).size()
    singleton_posts = n_per_post[n_per_post == 1].index

    df_clean = df_clean.loc[~df_clean[group_col].isin(singleton_posts)].copy()
    return df_clean

synapse_func_BC_clean  = clean_dropna_and_drop_singletons(synapse_func_BC)
synapse_func_BPC_clean = clean_dropna_and_drop_singletons(synapse_func_BPC)
synapse_func_MC_clean  = clean_dropna_and_drop_singletons(synapse_func_MC)
synapse_func_NGC_clean = clean_dropna_and_drop_singletons(synapse_func_NGC)

## Excitatory: Save functional df table

In [153]:
# === Read presynapse of excitatory neurons and save functional dataframe ===
units_23P = units_23P
fname_df = "23P"

# Read functional table to see the functional inputs
func_cols = ['pt_root_id', 'pref_ori', 'pref_dir', 'gOSI', 'gDSI', 'cc_abs']  # Only read columns we're interested in
func_props = (functional_properties_v3_bcm[func_cols].drop_duplicates(subset='pt_root_id'))

# Path
path_excitatory = Path(path_project) / "data/data_raw/synapse/"
path_save =  Path(path_project) / "results_excitatory"

# Loop over neurons and build one big dataframe for each typs of inhibitory neurons
all_synapse_func = []

for neuron in units_23P['pt_root_id']:
    fname = f"{neuron}_pre_synapse_df.pkl"
    fpath = path_excitatory / fname
    pre_synapse_df = pd.read_pickle(fpath)

    # Merge functional props onto each synapse with "pre_pt_root_id"
    pre_synapse_with_func = (pre_synapse_df.merge(
        func_props,
        left_on = "pre_pt_root_id",  # presynaptic neuron ID in synapse table
        right_on = "pt_root_id",     # same neuron ID in functional table
        how = "left")
    .drop(columns=["pt_root_id"])) # drop duplicate ID col from functional table

    # Keep only synapse id + funcitonal fields (and whatever else you want)
    synapse_func_df = pre_synapse_with_func[
        ['id', 'pre_pt_root_id', 'pref_ori', 'pref_dir', 'gOSI', 'gDSI', 'cc_abs']
    ].copy()

    # tag which postsyn neuron these synapses belong to
    synapse_func_df['post_pt_root_id'] = neuron
    all_synapse_func.append(synapse_func_df)

# Concatenate and save them as a pkl file
synapse_func_all = pd.concat(all_synapse_func, ignore_index=True)
out_path = path_save / f"synapse_func_all_{fname_df}.pkl"
synapse_func_all.to_pickle(out_path)
print(f"Saved {len(synapse_func_all)} synapses to {out_path}")

Saved 43350719 synapses to /media/DATA1/CK/python_script/MicronsBinder_04/results_excitatory/synapse_func_all_23P.pkl


## Excitatory: Read functional df table

In [8]:
path_read =  Path(path_project) / "results_excitatory"
synapse_func_23P = pd.read_pickle(path_read / "synapse_func_all_23P.pkl")

In [10]:
# Data Cleaning  - delete rows with no functional properties and delete neurons with only single connection

# Step 1) drop rows with missing functional props (pref_ori acts as the gate)
def clean_dropna_and_drop_singletons(df, group_col="post_pt_root_id", gate_col="pref_ori"):
    df_clean = df.loc[df[gate_col].notna()].copy()

    # Step 2) drop post neurons that have only a single remaining (non-NA) synapse row
    n_per_post = df_clean.groupby(group_col).size()
    singleton_posts = n_per_post[n_per_post == 1].index

    df_clean = df_clean.loc[~df_clean[group_col].isin(singleton_posts)].copy()
    return df_clean

synapse_func_23P_clean = clean_dropna_and_drop_singletons(synapse_func_23P)

## Plot Functions

### Basic plot (Plot all neurons)

In [37]:
# === Preferred orientation plot ===
def pref_ori_plot(
    synapse_func_all,
    neuron_type = "",
    bin_width_deg = 15,
    group_col = "post_pt_root_id", # per neuron
    save_opt = True,  # Boolean: true to save false to see the plot
    save_dir = "",
    ):

    """
        Plots preferred orientation distribution per each neuron
        and plot average of them
        Input:
            synapse_func_all: Insert concatenated dataframe of all neurons
    """

    # Define common bins for everyone
    bins = np.arange(0, 180 + bin_width_deg, bin_width_deg)
    bin_centers = (bins[:-1] + bins[1:]) / 2.0

    fig, ax = plt.subplots()
    all_y = []       # Each neuron's histogram

    # Loop over neurons
    for i, (neuron_id, df_neuron) in enumerate(synapse_func_all.groupby(group_col)):
        
        # Pull orientations, drop NaNs.
        ori_rad = df_neuron['pref_ori'].dropna().to_numpy()
        if ori_rad.size == 0:
            continue

        # Convert to degrees, wrap to [0, 180)
        ori_deg = np.degrees(ori_rad) % 180.0

        # Histogram for this neuron
        counts, _ = np.histogram(ori_deg, bins=bins)
        if counts.sum() == 0:
            continue
            
        y = counts
        all_y.append(y)

        # Plot this neuron's line in grey
        ax.plot(bin_centers, y, marker='o', linestyle='-', alpha=0.2, color='grey')

    if len(all_y) == 0:
        print("No valid neurons with pref_ori found")
        return

    # Compute mean across neurons and plot as black line
    mean_y = np.mean(np.stack(all_y), axis=0)
    
    ax.plot(bin_centers, mean_y, marker='o', linestyle='-', linewidth=2.5, color='black')
    ax.set_xlim(0, 180)
    ax.set_xticks(np.arange(0, 181, 30))
    ax.set_xlabel("preferred orientation (deg)")
    ax.set_ylabel("Number of synapses per bin")
    ax.set_title(f"Presynaptic orientation profiles ({neuron_type})")

    plt.tight_layout()

    # Decide to save or to just see
    if save_opt:
        
        # PNG
        fname = f"pref_ori_inhibitory_{neuron_type}.png"
        plt.savefig(os.path.join(save_dir, fname), dpi=300, format="png")

        # SVG
        fname = f"pref_ori_inhibitory_{neuron_type}.svg"
        plt.savefig(os.path.join(save_dir, fname), dpi=300, format="svg")

        plt.close(fig)
        
    else:
        plt.show()
        

In [38]:
# === Preferred direction plot ===
def pref_dir_plot(
    synapse_func_all,
    neuron_type = "",
    bin_width_deg=30,
    group_col="post_pt_root_id",  # per neuron
    save_opt=True,
    save_dir=""
):
    """
    Plots preferred direction distribution per each neuron
    and plots the average across neurons.
    Assumes pref_dir is in radians.
    """

    # Define common bins for everyone (0..360)
    bins = np.arange(0, 360 + bin_width_deg, bin_width_deg)
    bin_centers = (bins[:-1] + bins[1:]) / 2.0

    fig, ax = plt.subplots()
    all_y = []

    for neuron_id, df_neuron in synapse_func_all.groupby(group_col):
        dir_rad = df_neuron["pref_dir"].dropna().to_numpy()
        if dir_rad.size == 0:
            continue

        # Convert to degrees, wrap to [0, 360)
        dir_deg = np.degrees(dir_rad) % 360.0

        counts, _ = np.histogram(dir_deg, bins=bins)
        if counts.sum() == 0:
            continue

        y = counts
        all_y.append(y)

        ax.plot(bin_centers, y, marker="o", linestyle="-", alpha=0.2, color="grey")

    if len(all_y) == 0:
        print("No valid neurons with pref_dir found")
        return

    mean_y = np.mean(np.stack(all_y), axis=0)

    ax.plot(bin_centers, mean_y, marker="o", linestyle="-", linewidth=2.5, color="black")
    ax.set_xlim(0, 360)
    ax.set_xticks(np.arange(0, 361, 60))
    ax.set_xlabel("preferred direction (deg)")
    ax.set_ylabel("Number of synapses per bin")
    ax.set_title(f"Presynaptic direction profiles ({neuron_type})")

    plt.tight_layout()

    if save_opt:
        plt.savefig(os.path.join(save_dir, f"pref_dir_inhibitory_{neuron_type}.png"), dpi=300, format="png")
        plt.savefig(os.path.join(save_dir, f"pref_dir_inhibitory_{neuron_type}.svg"), dpi=300, format="svg")
        plt.close(fig)
    else:
        plt.show()


In [39]:
# === gOSI plot ===
def gosi_plot(
    synapse_func_all,
    neuron_type = "",
    bin_width=0.05,
    group_col="post_pt_root_id",  # per neuron
    save_opt=True,
    save_dir=""
):
    """
    Plots gOSI distribution per neuron (grey lines)
    and plots the mean across neurons (black line).
    """

    # Common bins for everyone (0..1)
    bins = np.arange(0.0, 1.0 + bin_width, bin_width)
    bin_centers = (bins[:-1] + bins[1:]) / 2.0

    fig, ax = plt.subplots()
    all_y = []

    for neuron_id, df_neuron in synapse_func_all.groupby(group_col):
        gosi = df_neuron["gOSI"].dropna().to_numpy()
        if gosi.size == 0:
            continue

        gosi = np.clip(gosi, 0.0, 1.0)

        counts, _ = np.histogram(gosi, bins=bins)
        if counts.sum() == 0:
            continue

        y = counts
        all_y.append(y)

        ax.plot(bin_centers, y, marker="o", linestyle="-", alpha=0.2, color="grey")

    if len(all_y) == 0:
        print("No valid neurons with gOSI found")
        return

    mean_y = np.mean(np.stack(all_y), axis=0)

    ax.plot(bin_centers, mean_y, marker="o", linestyle="-", linewidth=2.5, color="black")
    ax.set_xlim(0, 1)
    ax.set_xlabel("gOSI")
    ax.set_ylabel("Number of synapses per bin")
    ax.set_title(f"Presynaptic gOSI profiles ({neuron_type})")

    plt.tight_layout()

    if save_opt:
        plt.savefig(os.path.join(save_dir, f"gOSI_inhibitory_{neuron_type}.png"), dpi=300, format="png")
        plt.savefig(os.path.join(save_dir, f"gOSI_inhibitory_{neuron_type}.svg"), dpi=300, format="svg")
        plt.close(fig)
    else:
        plt.show()


In [40]:
# === gDSI plot ===
def gdsi_plot(
    synapse_func_all,
    neuron_type = "",
    bin_width=0.05,
    group_col="post_pt_root_id",  # per neuron
    save_opt=True,
    save_dir=""
):
    """
    Plots gDSI distribution per neuron (grey lines)
    and plots the mean across neurons (black line).
    """

    # Common bins for everyone (0..1)
    bins = np.arange(0.0, 1.0 + bin_width, bin_width)
    bin_centers = (bins[:-1] + bins[1:]) / 2.0

    fig, ax = plt.subplots()
    all_y = []

    for neuron_id, df_neuron in synapse_func_all.groupby(group_col):
        gdsi = df_neuron["gDSI"].dropna().to_numpy()
        if gdsi.size == 0:
            continue

        gdsi = np.clip(gdsi, 0.0, 1.0)

        counts, _ = np.histogram(gdsi, bins=bins)
        if counts.sum() == 0:
            continue

        y = counts
        all_y.append(y)

        ax.plot(bin_centers, y, marker="o", linestyle="-", alpha=0.2, color="grey")

    if len(all_y) == 0:
        print("No valid neurons with gDSI found")
        return

    mean_y = np.mean(np.stack(all_y), axis=0)

    ax.plot(bin_centers, mean_y, marker="o", linestyle="-", linewidth=2.5, color="black")
    ax.set_xlim(0, 1)
    ax.set_xlabel("gDSI")
    ax.set_ylabel("Number of synapses per bin")
    ax.set_title(f"Presynaptic gDSI profiles ({neuron_type})")

    plt.tight_layout()

    if save_opt:
        plt.savefig(os.path.join(save_dir, f"gDSI_inhibitory_{neuron_type}.png"), dpi=300, format="png")
        plt.savefig(os.path.join(save_dir, f"gDSI_inhibitory_{neuron_type}.svg"), dpi=300, format="svg")
        plt.close(fig)
    else:
        plt.show()


In [41]:
# === cc_abs plot ===
def cc_abs_plot(
    synapse_func_all,
    neuron_type = "",
    bin_width=0.05,
    group_col="post_pt_root_id",  # per neuron
    save_opt=True,
    save_dir=""
):
    """
    Plots cc_abs distribution per neuron (grey lines)
    and plots the mean across neurons (black line).
    Assumes cc_abs is in [0, 1] (clips just in case).
    """

    bins = np.arange(0.0, 1.0 + bin_width, bin_width)
    bin_centers = (bins[:-1] + bins[1:]) / 2.0

    fig, ax = plt.subplots()
    all_y = []

    for neuron_id, df_neuron in synapse_func_all.groupby(group_col):
        cc = df_neuron["cc_abs"].dropna().to_numpy()
        if cc.size == 0:
            continue

        cc = np.clip(cc, 0.0, 1.0)

        counts, _ = np.histogram(cc, bins=bins)
        if counts.sum() == 0:
            continue

        y = counts
        all_y.append(y)

        ax.plot(bin_centers, y, marker="o", linestyle="-", alpha=0.2, color="grey")

    if len(all_y) == 0:
        print("No valid neurons with cc_abs found")
        return

    mean_y = np.mean(np.stack(all_y), axis=0)

    ax.plot(bin_centers, mean_y, marker="o", linestyle="-", linewidth=2.5, color="black")
    ax.set_xlim(0, 1)
    ax.set_xlabel("cc_abs")
    ax.set_ylabel("Number of synapses per bin")
    ax.set_title(f"Presynaptic cc_abs profiles ({neuron_type})")

    plt.tight_layout()

    if save_opt:
        plt.savefig(os.path.join(save_dir, f"cc_abs_inhibitory_{neuron_type}.png"), dpi=300, format="png")
        plt.savefig(os.path.join(save_dir, f"cc_abs_inhibitory_{neuron_type}.svg"), dpi=300, format="svg")
        plt.close(fig)
    else:
        plt.show()


### Plot peaks of histogram (single)

In [75]:
# === Extract a peak point for each neuron's presynaptic orientation distribution

# Extract peaks of pref_ori
def extract_pref_ori_peaks(synapse_func_all, bin_width_deg=15, group_col="post_pt_root_id"):

    bins = np.arange(0, 180 + bin_width_deg, bin_width_deg)
    bin_centers = (bins[:-1] + bins[1:]) / 2.0

    peak_x = []
    peak_y = []
    neuron_ids = []

    for neuron_id, df_neuron in synapse_func_all.groupby(group_col):
        ori_rad = df_neuron["pref_ori"].dropna().to_numpy()
        if ori_rad.size == 0:
            continue

        ori_deg = np.degrees(ori_rad) % 180.0
        counts, _ = np.histogram(ori_deg, bins=bins)
        if counts.sum() == 0:
            continue

        max_count = counts.max()
        peak_idxs = np.flatnonzero(counts == max_count)

        # Axial circular mean of tied peak bin centers (handles 0/180 wrap correctly)
        peak_centers = bin_centers[peak_idxs]
        ang = np.deg2rad(peak_centers)

        # Double-angle trick for orientation (period = pi)
        ang2 = 2.0 * ang
        mean_ang2 = np.arctan2(np.mean(np.sin(ang2)), np.mean(np.cos(ang2)))
        peak_center = float((np.rad2deg(mean_ang2) / 2.0) % 180.0)

        neuron_ids.append(neuron_id)
        peak_x.append(peak_center)
        peak_y.append(max_count)

    return np.array(neuron_ids), np.array(peak_x), np.array(peak_y), bins

# Extract peaks of pref_dir
def extract_pref_dir_peaks(synapse_func_all, bin_width_deg=30, group_col="post_pt_root_id"):

    bins = np.arange(0, 360 + bin_width_deg, bin_width_deg)
    bin_centers = (bins[:-1] + bins[1:]) / 2.0

    peak_x = []
    peak_y = []
    neuron_ids = []

    for neuron_id, df_neuron in synapse_func_all.groupby(group_col):
        dir_rad = df_neuron["pref_dir"].dropna().to_numpy()
        if dir_rad.size == 0:
            continue

        dir_deg = np.degrees(dir_rad) % 360.0
        counts, _ = np.histogram(dir_deg, bins=bins)
        if counts.sum() == 0:
            continue

        max_count = counts.max()
        peak_idxs = np.flatnonzero(counts == max_count)

        # Circular-mean of tied peak bin centers (handles 0/360 wrap correctly)
        peak_centers = bin_centers[peak_idxs]
        ang = np.deg2rad(peak_centers)
        mean_ang = np.arctan2(np.mean(np.sin(ang)), np.mean(np.cos(ang)))
        peak_center = float(np.rad2deg(mean_ang) % 360.0)

        neuron_ids.append(neuron_id)
        peak_x.append(peak_center)
        peak_y.append(max_count)

    return np.array(neuron_ids), np.array(peak_x), np.array(peak_y), bins

# Extract gosi peaks
def extract_gosi_peaks(synapse_func_all, bin_width=0.05, group_col="post_pt_root_id"):

    bins = np.arange(0.0, 1.0 + bin_width, bin_width)
    bin_centers = (bins[:-1] + bins[1:]) / 2.0

    peak_x = []
    peak_y = []
    neuron_ids = []

    for neuron_id, df_neuron in synapse_func_all.groupby(group_col):
        gosi = df_neuron["gOSI"].dropna().to_numpy()
        if gosi.size == 0:
            continue

        gosi = np.clip(gosi, 0.0, 1.0)

        counts, _ = np.histogram(gosi, bins=bins)
        if counts.sum() == 0:
            continue

        # handle ties by taking mean of tied peak bins
        max_count = counts.max()
        peak_idxs = np.flatnonzero(counts == max_count)
        peak_center = float(np.mean(bin_centers[peak_idxs]))

        neuron_ids.append(neuron_id)
        peak_x.append(peak_center)
        peak_y.append(max_count)

    return np.array(neuron_ids), np.array(peak_x), np.array(peak_y), bins

# Extract gdsi peaks
def extract_gdsi_peaks(synapse_func_all, bin_width=0.05, group_col="post_pt_root_id"):

    bins = np.arange(0.0, 1.0 + bin_width, bin_width)
    bin_centers = (bins[:-1] + bins[1:]) / 2.0

    peak_x = []
    peak_y = []
    neuron_ids = []

    for neuron_id, df_neuron in synapse_func_all.groupby(group_col):
        gdsi = df_neuron["gDSI"].dropna().to_numpy()
        if gdsi.size == 0:
            continue

        gdsi = np.clip(gdsi, 0.0, 1.0)

        counts, _ = np.histogram(gdsi, bins=bins)
        if counts.sum() == 0:
            continue

        # handle ties by taking mean of tied peak bins
        max_count = counts.max()
        peak_idxs = np.flatnonzero(counts == max_count)
        peak_center = float(np.mean(bin_centers[peak_idxs]))

        neuron_ids.append(neuron_id)
        peak_x.append(peak_center)
        peak_y.append(max_count)

    return np.array(neuron_ids), np.array(peak_x), np.array(peak_y), bins

# Extract cc_abs peaks
def extract_cc_abs_peaks(synapse_func_all, bin_width=0.05, group_col="post_pt_root_id"):

    bins = np.arange(0.0, 1.0 + bin_width, bin_width)
    bin_centers = (bins[:-1] + bins[1:]) / 2.0

    peak_x = []
    peak_y = []
    neuron_ids = []

    for neuron_id, df_neuron in synapse_func_all.groupby(group_col):
        cc = df_neuron["cc_abs"].dropna().to_numpy()
        if cc.size == 0:
            continue

        cc = np.clip(cc, 0.0, 1.0)

        counts, _ = np.histogram(cc, bins=bins)
        if counts.sum() == 0:
            continue

        # handle ties by taking mean of tied peak bins
        max_count = counts.max()
        peak_idxs = np.flatnonzero(counts == max_count)
        peak_center = float(np.mean(bin_centers[peak_idxs]))

        neuron_ids.append(neuron_id)
        peak_x.append(peak_center)
        peak_y.append(max_count)

    return np.array(neuron_ids), np.array(peak_x), np.array(peak_y), bins

In [76]:
# === Preferred orientation peak distribution plot ===
def plot_pref_ori_peak_distribution_single(
    synapse_func_all,
    neuron_type="",
    bin_width_deg=15,
    group_col="post_pt_root_id",
    save_opt = True,  # Boolean: true to save false to see the plot
    save_dir = "",
):

    neuron_ids, peak_x, peak_y, bins = extract_pref_ori_peaks(
        synapse_func_all, bin_width_deg=bin_width_deg, group_col=group_col
    )

    if peak_x.size == 0:
        print("No valid neurons with pref_ori found")
        return

    # --- Figure: distribution of peak locations across neurons ---
    fig, ax = plt.subplots()
    
    ax.hist(peak_x, bins=bins, color = "grey", edgecolor="black", label=f"{neuron_type} (n={peak_x.size})")
    ax.set_ylabel("number of neurons")
    ax.set_title(f"Distribution of peak orientations ({neuron_type})")

    ax.set_xlim(0, 180)
    ax.set_xticks(np.arange(0, 181, 30))
    ax.set_xlabel("peak preferred orientation (deg)")
    ax.legend(frameon=False, loc="upper right")

    plt.tight_layout()

    if save_opt:
        plt.savefig(os.path.join(save_dir, f"pref_ori_peak_{neuron_type}.png"), dpi=300, format="png")
        plt.savefig(os.path.join(save_dir, f"pref_ori_peak_{neuron_type}.svg"), dpi=300, format="svg")

        plt.close(fig)
    else:
        plt.show()


In [77]:
# === Preferred direction peak distribution plot ===
def plot_pref_dir_peak_distribution_single(
    synapse_func_all,
    neuron_type="",
    bin_width_deg=30,
    group_col="post_pt_root_id",
    save_opt=True,
    save_dir="",
):

    neuron_ids, peak_x, peak_y, bins = extract_pref_dir_peaks(
        synapse_func_all, bin_width_deg=bin_width_deg, group_col=group_col
    )

    if peak_x.size == 0:
        print("No valid neurons with pref_dir found")
        return

    fig, ax = plt.subplots()

    ax.hist(peak_x, bins=bins, color = "grey", edgecolor="black", label=f"{neuron_type} (n={peak_x.size})")
    ax.set_ylabel("number of neurons")
    ax.set_title(f"Distribution of peak directions ({neuron_type})")

    ax.set_xlim(0, 360)
    ax.set_xticks(np.arange(0, 361, 60))
    ax.set_xlabel("peak preferred direction (deg)")
    ax.legend(frameon=False, loc="upper right")

    plt.tight_layout()

    if save_opt:
        plt.savefig(os.path.join(save_dir, f"pref_dir_peak_{neuron_type}.png"), dpi=300, format="png")
        plt.savefig(os.path.join(save_dir, f"pref_dir_peak_{neuron_type}.svg"), dpi=300, format="svg")
        plt.close(fig)
    else:
        plt.show()


In [78]:
# === gOSI peak distribution plot ===
def plot_gosi_peak_distribution_single(
    synapse_func_all,
    neuron_type="",
    bin_width=0.05,
    group_col="post_pt_root_id",
    save_opt=True,
    save_dir="",
):

    neuron_ids, peak_x, peak_y, bins = extract_gosi_peaks(
        synapse_func_all, bin_width=bin_width, group_col=group_col
    )

    if peak_x.size == 0:
        print("No valid neurons with gOSI found")
        return

    fig, ax = plt.subplots()

    ax.hist(peak_x, bins=bins, color="grey", edgecolor="black", label=f"{neuron_type} (n={peak_x.size})")
    ax.set_xlim(0, 1)
    ax.set_xlabel("peak gOSI")
    ax.set_ylabel("number of neurons")
    ax.set_title(f"Distribution of peak gOSI ({neuron_type})")
    ax.legend(frameon=False, loc="upper right")

    plt.tight_layout()

    if save_opt:
        plt.savefig(os.path.join(save_dir, f"gOSI_peak_{neuron_type}.png"), dpi=300, format="png")
        plt.savefig(os.path.join(save_dir, f"gOSI_peak_{neuron_type}.svg"), dpi=300, format="svg")
        plt.close(fig)
    else:
        plt.show()


In [79]:
# === gDSI peak distribution plot ===
def plot_gdsi_peak_distribution_single(
    synapse_func_all,
    neuron_type="",
    bin_width=0.05,
    group_col="post_pt_root_id",
    save_opt=True,
    save_dir="",
):

    neuron_ids, peak_x, peak_y, bins = extract_gdsi_peaks(
        synapse_func_all, bin_width=bin_width, group_col=group_col
    )

    if peak_x.size == 0:
        print("No valid neurons with gDSI found")
        return

    fig, ax = plt.subplots()

    ax.hist(peak_x, bins=bins, color = "grey", edgecolor="black", label=f"{neuron_type} (n={peak_x.size})")
    ax.set_xlim(0, 1)
    ax.set_xlabel("peak gDSI")
    ax.set_ylabel("number of neurons")
    ax.set_title(f"Distribution of peak gDSI ({neuron_type})")
    ax.legend(frameon=False, loc="upper right")

    plt.tight_layout()

    if save_opt:
        plt.savefig(os.path.join(save_dir, f"gDSI_peak_{neuron_type}.png"), dpi=300, format="png")
        plt.savefig(os.path.join(save_dir, f"gDSI_peak_{neuron_type}.svg"), dpi=300, format="svg")
        plt.close(fig)
    else:
        plt.show()


In [80]:
# === cc_abs peak distribution plot ===
def plot_cc_abs_peak_distribution_single(
    synapse_func_all,
    neuron_type="",
    bin_width=0.05,
    group_col="post_pt_root_id",
    save_opt=True,
    save_dir="",
):

    neuron_ids, peak_x, peak_y, bins = extract_cc_abs_peaks(
        synapse_func_all, bin_width=bin_width, group_col=group_col
    )

    if peak_x.size == 0:
        print("No valid neurons with cc_abs found")
        return

    fig, ax = plt.subplots()

    ax.hist(peak_x, bins=bins, color = "grey", edgecolor="black", label=f"{neuron_type} (n={peak_x.size})")
    ax.set_xlim(0, 1)
    ax.set_xlabel("peak cc_abs")
    ax.set_ylabel("number of neurons")
    ax.set_title(f"Distribution of peak cc_abs ({neuron_type})")
    ax.legend(frameon=False, loc="upper right")

    plt.tight_layout()

    if save_opt:
        plt.savefig(os.path.join(save_dir, f"cc_abs_peak_{neuron_type}.png"), dpi=300, format="png")
        plt.savefig(os.path.join(save_dir, f"cc_abs_peak_{neuron_type}.svg"), dpi=300, format="svg")
        plt.close(fig)
    else:
        plt.show()


### Plot peaks of histogram (overlay)

In [132]:
# === Helpers ===

def _gaussian_kernel1d(sigma_bins, radius=4):
    """Gaussian kernel in *bin units*."""
    if sigma_bins <= 0:
        return np.array([1.0])
    r = int(np.ceil(radius * sigma_bins))
    x = np.arange(-r, r + 1)
    k = np.exp(-(x**2) / (2 * sigma_bins**2))
    k /= k.sum()
    return k

def _extract_peak_x_numeric(
    synapse_func_all,
    value_col,
    bins,
    bin_centers,
    group_col="post_pt_root_id",
    clip_range=None,   # (lo, hi) or None
    to_degrees=False,  # for pref_dir
    mod_val=None,      # 360 for pref_dir
):
    """
    Returns one peak bin center per neuron for a numeric feature.
    Peak is defined as the modal bin of that neuron's values.
    """
    peak_x = []

    for neuron_id, df_neuron in synapse_func_all.groupby(group_col):
        vals = df_neuron[value_col].dropna().to_numpy()
        if vals.size == 0:
            continue

        if to_degrees:
            vals = np.degrees(vals)
        if mod_val is not None:
            vals = vals % mod_val
        if clip_range is not None:
            lo, hi = clip_range
            vals = np.clip(vals, lo, hi)

        counts, _ = np.histogram(vals, bins=bins)
        if counts.sum() == 0:
            continue

        max_count = counts.max()
        peak_idxs = np.flatnonzero(counts == max_count)
        peak_center = float(np.mean(bin_centers[peak_idxs]))  # average ties
        peak_x.append(peak_center)

    return np.array(peak_x)


In [134]:
# === OVERLAYED PLOT: Preferred orientation peak distribution plot ===

def plot_pref_ori_peak_distribution(
    synapse_func_dict,            # {"BC": df, "BPC": df, "MC": df, "NGC": df}
    synapse_func_exc=None,        # excitatory df (e.g., synapse_func_23P_clean) or None
    exc_label="Exc",
    bin_width_deg=15,
    group_col="post_pt_root_id",
    density=False,
    alpha=0.8,
    linewidth=2.0,
    save_opt=True,
    save_dir="",
    fname="pref_ori_peak_overlay",
):
    """
    Overlay hollow step-hist silhouettes of per-neuron peak preferred ORIENTATION (0..180)
    for inhibitory groups (+ optional excitatory overlay in black).
    """

    bins = np.arange(0, 180 + bin_width_deg, bin_width_deg)
    bin_centers = (bins[:-1] + bins[1:]) / 2.0

    def _extract_peak_x(synapse_func_all):
        peak_x = []
        for neuron_id, df_neuron in synapse_func_all.groupby(group_col):
            ori_rad = df_neuron["pref_ori"].dropna().to_numpy()
            if ori_rad.size == 0:
                continue

            ori_deg = np.degrees(ori_rad) % 180.0
            counts, _ = np.histogram(ori_deg, bins=bins)
            if counts.sum() == 0:
                continue

            max_count = counts.max()
            peak_idxs = np.flatnonzero(counts == max_count)
            peak_center = float(np.mean(bin_centers[peak_idxs]))
            peak_x.append(peak_center)
        return np.array(peak_x)

    fig, ax = plt.subplots()

    # --- inhibitory groups (default color cycle) ---
    for label, synapse_func_all in synapse_func_dict.items():
        peak_x = _extract_peak_x(synapse_func_all)
        if peak_x.size == 0:
            print(f"[{label}] No valid neurons with pref_ori peaks found.")
            continue

        ax.hist(
            peak_x,
            bins=bins,
            density=density,
            histtype="step",
            linewidth=linewidth,
            alpha=alpha,
            label=f"{label} (n={peak_x.size})",
        )

    # --- excitatory overlay in black ---
    if synapse_func_exc is not None:
        peak_x_exc = _extract_peak_x(synapse_func_exc)
        if peak_x_exc.size == 0:
            print(f"[{exc_label}] No valid neurons with pref_ori peaks found.")
        else:
            ax.hist(
                peak_x_exc,
                bins=bins,
                density=density,
                histtype="step",
                linewidth=linewidth + 0.5,   # slightly bolder so it reads on top
                alpha=1.0,
                color="black",
                label=f"{exc_label} (n={peak_x_exc.size})",
                zorder=10,
            )

    ax.set_xlim(0, 180)
    ax.set_xticks(np.arange(0, 181, 30))
    ax.set_xlabel("peak preferred orientation (deg)")
    ax.set_ylabel("density" if density else "number of neurons")
    ax.set_title("Distribution of peak orientations (overlay)")
    ax.legend(frameon=False)

    plt.tight_layout()

    if save_opt:
        plt.savefig(os.path.join(save_dir, f"{fname}.png"), dpi=300, format="png")
        plt.savefig(os.path.join(save_dir, f"{fname}.svg"), dpi=300, format="svg")
        plt.close(fig)
    else:
        plt.show()

# === OVERLAYED PLOT: CURVED ===
def plot_pref_ori_peak_distribution_curve(
    synapse_func_dict,            # {"BC": df, ...}
    synapse_func_exc=None,        # excitatory df or None
    exc_label="Exc",
    bin_width_deg=15,
    group_col="post_pt_root_id",
    density=True,
    smooth_sigma_bins=1.0,        # smoothing strength in *bins* (try 0.8–1.5)
    alpha=0.9,
    linewidth=2.5,
    save_opt=True,
    save_dir="",
    fname="pref_ori_peak_overlay_curve",
):
    """
    Overlay *smoothed* curves of per-neuron peak preferred orientation (0..180).
    Uses a histogram on common bins then Gaussian-smooths across bins.
    """

    bins = np.arange(0, 180 + bin_width_deg, bin_width_deg)
    bin_centers = (bins[:-1] + bins[1:]) / 2.0

    def _extract_peak_x(synapse_func_all):
        peak_x = []
        for neuron_id, df_neuron in synapse_func_all.groupby(group_col):
            ori_rad = df_neuron["pref_ori"].dropna().to_numpy()
            if ori_rad.size == 0:
                continue
            ori_deg = np.degrees(ori_rad) % 180.0
            counts, _ = np.histogram(ori_deg, bins=bins)
            if counts.sum() == 0:
                continue
            max_count = counts.max()
            peak_idxs = np.flatnonzero(counts == max_count)
            peak_center = float(np.mean(bin_centers[peak_idxs]))
            peak_x.append(peak_center)
        return np.array(peak_x)

    kernel = _gaussian_kernel1d(sigma_bins=smooth_sigma_bins)

    fig, ax = plt.subplots()

    # inhibitory curves
    for label, synapse_func_all in synapse_func_dict.items():
        peak_x = _extract_peak_x(synapse_func_all)
        if peak_x.size == 0:
            print(f"[{label}] No valid neurons with pref_ori peaks found.")
            continue

        y, _ = np.histogram(peak_x, bins=bins, density=density)
        y_smooth = np.convolve(y, kernel, mode="same")

        ax.plot(
            bin_centers, y_smooth,
            alpha=alpha,
            linewidth=linewidth,
            label=f"{label} (n={peak_x.size})",
        )

    # excitatory curve in black
    if synapse_func_exc is not None:
        peak_x_exc = _extract_peak_x(synapse_func_exc)
        if peak_x_exc.size == 0:
            print(f"[{exc_label}] No valid neurons with pref_ori peaks found.")
        else:
            y_exc, _ = np.histogram(peak_x_exc, bins=bins, density=density)
            y_exc_smooth = np.convolve(y_exc, kernel, mode="same")
            ax.plot(
                bin_centers, y_exc_smooth,
                color="black",
                alpha=1.0,
                linewidth=linewidth + 0.5,
                label=f"{exc_label} (n={peak_x_exc.size})",
                zorder=10,
            )

    ax.set_xlim(0, 180)
    ax.set_xticks(np.arange(0, 181, 30))
    ax.set_xlabel("peak preferred orientation (deg)")
    ax.set_ylabel("density" if density else "number of neurons per bin")
    ax.set_title("Peak orientation distribution (smoothed curves)")
    ax.legend(frameon=False)

    plt.tight_layout()

    if save_opt:
        plt.savefig(os.path.join(save_dir, f"{fname}.png"), dpi=300, format="png")
        plt.savefig(os.path.join(save_dir, f"{fname}.svg"), dpi=300, format="svg")
        plt.close(fig)
    else:
        plt.show()

In [135]:
# === OVERLAYED PLOT Preferred direction peak distribution plot ===

def plot_pref_dir_peak_distribution_step(
    synapse_func_dict,
    synapse_func_exc=None,
    exc_label="Exc",
    bin_width_deg=30,
    group_col="post_pt_root_id",
    density=True,
    alpha=0.8,
    linewidth=2.0,
    save_opt=True,
    save_dir="",
    fname="pref_dir_peak_overlay_step",
):
    bins = np.arange(0, 360 + bin_width_deg, bin_width_deg)
    bin_centers = (bins[:-1] + bins[1:]) / 2.0

    fig, ax = plt.subplots()

    for label, synapse_func_all in synapse_func_dict.items():
        peak_x = _extract_peak_x_numeric(
            synapse_func_all,
            value_col="pref_dir",
            bins=bins,
            bin_centers=bin_centers,
            group_col=group_col,
            to_degrees=True,
            mod_val=360.0,
        )
        if peak_x.size == 0:
            print(f"[{label}] No valid neurons with pref_dir peaks found.")
            continue

        ax.hist(
            peak_x, bins=bins, density=density,
            histtype="step", linewidth=linewidth, alpha=alpha,
            label=f"{label} (n={peak_x.size})",
        )

    if synapse_func_exc is not None:
        peak_x_exc = _extract_peak_x_numeric(
            synapse_func_exc,
            value_col="pref_dir",
            bins=bins,
            bin_centers=bin_centers,
            group_col=group_col,
            to_degrees=True,
            mod_val=360.0,
        )
        if peak_x_exc.size > 0:
            ax.hist(
                peak_x_exc, bins=bins, density=density,
                histtype="step", linewidth=linewidth + 0.5, alpha=1.0,
                color="black", zorder=10,
                label=f"{exc_label} (n={peak_x_exc.size})",
            )

    ax.set_xlim(0, 360)
    ax.set_xticks(np.arange(0, 361, 60))
    ax.set_xlabel("peak preferred direction (deg)")
    ax.set_ylabel("density" if density else "number of neurons")
    ax.set_title("Distribution of peak directions (overlay)")
    ax.legend(frameon=False)
    plt.tight_layout()

    if save_opt:
        plt.savefig(os.path.join(save_dir, f"{fname}.png"), dpi=300)
        plt.savefig(os.path.join(save_dir, f"{fname}.svg"), dpi=300)
        plt.close(fig)
    else:
        plt.show()

# === CURVED VERSION: OVERLAYED PLOT===
def plot_pref_dir_peak_distribution_curve(
    synapse_func_dict,
    synapse_func_exc=None,
    exc_label="Exc",
    bin_width_deg=30,
    group_col="post_pt_root_id",
    density=True,
    smooth_sigma_bins=1.0,
    alpha=0.9,
    linewidth=2.5,
    save_opt=True,
    save_dir="",
    fname="pref_dir_peak_overlay_curve",
):
    bins = np.arange(0, 360 + bin_width_deg, bin_width_deg)
    bin_centers = (bins[:-1] + bins[1:]) / 2.0
    kernel = _gaussian_kernel1d(sigma_bins=smooth_sigma_bins)

    fig, ax = plt.subplots()

    for label, synapse_func_all in synapse_func_dict.items():
        peak_x = _extract_peak_x_numeric(
            synapse_func_all, "pref_dir", bins, bin_centers,
            group_col=group_col, to_degrees=True, mod_val=360.0
        )
        if peak_x.size == 0:
            continue

        y, _ = np.histogram(peak_x, bins=bins, density=density)
        y_s = np.convolve(y, kernel, mode="same")
        ax.plot(bin_centers, y_s, alpha=alpha, linewidth=linewidth, label=f"{label} (n={peak_x.size})")

    if synapse_func_exc is not None:
        peak_x_exc = _extract_peak_x_numeric(
            synapse_func_exc, "pref_dir", bins, bin_centers,
            group_col=group_col, to_degrees=True, mod_val=360.0
        )
        if peak_x_exc.size > 0:
            y, _ = np.histogram(peak_x_exc, bins=bins, density=density)
            y_s = np.convolve(y, kernel, mode="same")
            ax.plot(bin_centers, y_s, color="black", alpha=1.0,
                    linewidth=linewidth + 0.5, zorder=10, label=f"{exc_label} (n={peak_x_exc.size})")

    ax.set_xlim(0, 360)
    ax.set_xticks(np.arange(0, 361, 60))
    ax.set_xlabel("peak preferred direction (deg)")
    ax.set_ylabel("density" if density else "number of neurons per bin")
    ax.set_title("Peak direction distribution (smoothed curves)")
    ax.legend(frameon=False)
    plt.tight_layout()

    if save_opt:
        plt.savefig(os.path.join(save_dir, f"{fname}.png"), dpi=300)
        plt.savefig(os.path.join(save_dir, f"{fname}.svg"), dpi=300)
        plt.close(fig)
    else:
        plt.show()


In [136]:
# === OVERLAYED PLOT gOSI peak distribution plot ===
def plot_gosi_peak_distribution_step(
    synapse_func_dict,
    synapse_func_exc=None,
    exc_label="Exc",
    bin_width=0.05,
    group_col="post_pt_root_id",
    density=True,
    alpha=0.8,
    linewidth=2.0,
    save_opt=True,
    save_dir="",
    fname="gOSI_peak_overlay_step",
):
    bins = np.arange(0.0, 1.0 + bin_width, bin_width)
    bin_centers = (bins[:-1] + bins[1:]) / 2.0

    fig, ax = plt.subplots()

    for label, synapse_func_all in synapse_func_dict.items():
        peak_x = _extract_peak_x_numeric(
            synapse_func_all, "gOSI", bins, bin_centers,
            group_col=group_col, clip_range=(0.0, 1.0)
        )
        if peak_x.size == 0:
            continue
        ax.hist(peak_x, bins=bins, density=density, histtype="step",
                linewidth=linewidth, alpha=alpha, label=f"{label} (n={peak_x.size})")

    if synapse_func_exc is not None:
        peak_x_exc = _extract_peak_x_numeric(
            synapse_func_exc, "gOSI", bins, bin_centers,
            group_col=group_col, clip_range=(0.0, 1.0)
        )
        if peak_x_exc.size > 0:
            ax.hist(peak_x_exc, bins=bins, density=density, histtype="step",
                    linewidth=linewidth + 0.5, alpha=1.0, color="black", zorder=10,
                    label=f"{exc_label} (n={peak_x_exc.size})")

    ax.set_xlim(0, 1)
    ax.set_xlabel("peak gOSI")
    ax.set_ylabel("density" if density else "number of neurons")
    ax.set_title("Distribution of peak gOSI (overlay)")
    ax.legend(frameon=False)
    plt.tight_layout()

    if save_opt:
        plt.savefig(os.path.join(save_dir, f"{fname}.png"), dpi=300)
        plt.savefig(os.path.join(save_dir, f"{fname}.svg"), dpi=300)
        plt.close(fig)
    else:
        plt.show()

# === CURVED: OVERLAYED ===
def plot_gosi_peak_distribution_curve(
    synapse_func_dict,
    synapse_func_exc=None,
    exc_label="Exc",
    bin_width=0.05,
    group_col="post_pt_root_id",
    density=True,
    smooth_sigma_bins=1.0,
    alpha=0.9,
    linewidth=2.5,
    save_opt=True,
    save_dir="",
    fname="gOSI_peak_overlay_curve",
):
    bins = np.arange(0.0, 1.0 + bin_width, bin_width)
    bin_centers = (bins[:-1] + bins[1:]) / 2.0
    kernel = _gaussian_kernel1d(sigma_bins=smooth_sigma_bins)

    fig, ax = plt.subplots()

    for label, synapse_func_all in synapse_func_dict.items():
        peak_x = _extract_peak_x_numeric(
            synapse_func_all, "gOSI", bins, bin_centers,
            group_col=group_col, clip_range=(0.0, 1.0)
        )
        if peak_x.size == 0:
            continue
        y, _ = np.histogram(peak_x, bins=bins, density=density)
        ax.plot(bin_centers, np.convolve(y, kernel, mode="same"),
                alpha=alpha, linewidth=linewidth, label=f"{label} (n={peak_x.size})")

    if synapse_func_exc is not None:
        peak_x_exc = _extract_peak_x_numeric(
            synapse_func_exc, "gOSI", bins, bin_centers,
            group_col=group_col, clip_range=(0.0, 1.0)
        )
        if peak_x_exc.size > 0:
            y, _ = np.histogram(peak_x_exc, bins=bins, density=density)
            ax.plot(bin_centers, np.convolve(y, kernel, mode="same"),
                    color="black", alpha=1.0, linewidth=linewidth + 0.5, zorder=10,
                    label=f"{exc_label} (n={peak_x_exc.size})")

    ax.set_xlim(0, 1)
    ax.set_xlabel("peak gOSI")
    ax.set_ylabel("density" if density else "number of neurons per bin")
    ax.set_title("Peak gOSI distribution (smoothed curves)")
    ax.legend(frameon=False)
    plt.tight_layout()

    if save_opt:
        plt.savefig(os.path.join(save_dir, f"{fname}.png"), dpi=300)
        plt.savefig(os.path.join(save_dir, f"{fname}.svg"), dpi=300)
        plt.close(fig)
    else:
        plt.show()


In [137]:
# === OVERLAYED PLOT gDSI peak distribution plot ===
def plot_gdsi_peak_distribution_step(
    synapse_func_dict,
    synapse_func_exc=None,
    exc_label="Exc",
    bin_width=0.05,
    group_col="post_pt_root_id",
    density=True,
    alpha=0.8,
    linewidth=2.0,
    save_opt=True,
    save_dir="",
    fname="gDSI_peak_overlay_step",
):
    bins = np.arange(0.0, 1.0 + bin_width, bin_width)
    bin_centers = (bins[:-1] + bins[1:]) / 2.0

    fig, ax = plt.subplots()

    for label, synapse_func_all in synapse_func_dict.items():
        peak_x = _extract_peak_x_numeric(
            synapse_func_all, "gDSI", bins, bin_centers,
            group_col=group_col, clip_range=(0.0, 1.0)
        )
        if peak_x.size == 0:
            continue
        ax.hist(peak_x, bins=bins, density=density, histtype="step",
                linewidth=linewidth, alpha=alpha, label=f"{label} (n={peak_x.size})")

    if synapse_func_exc is not None:
        peak_x_exc = _extract_peak_x_numeric(
            synapse_func_exc, "gDSI", bins, bin_centers,
            group_col=group_col, clip_range=(0.0, 1.0)
        )
        if peak_x_exc.size > 0:
            ax.hist(peak_x_exc, bins=bins, density=density, histtype="step",
                    linewidth=linewidth + 0.5, alpha=1.0, color="black", zorder=10,
                    label=f"{exc_label} (n={peak_x_exc.size})")

    ax.set_xlim(0, 1)
    ax.set_xlabel("peak gDSI")
    ax.set_ylabel("density" if density else "number of neurons")
    ax.set_title("Distribution of peak gDSI (overlay)")
    ax.legend(frameon=False)
    plt.tight_layout()

    if save_opt:
        plt.savefig(os.path.join(save_dir, f"{fname}.png"), dpi=300)
        plt.savefig(os.path.join(save_dir, f"{fname}.svg"), dpi=300)
        plt.close(fig)
    else:
        plt.show()

# === CURVED: OVERLAYED PLOT ===

def plot_gdsi_peak_distribution_curve(
    synapse_func_dict,
    synapse_func_exc=None,
    exc_label="Exc",
    bin_width=0.05,
    group_col="post_pt_root_id",
    density=True,
    smooth_sigma_bins=1.0,
    alpha=0.9,
    linewidth=2.5,
    save_opt=True,
    save_dir="",
    fname="gDSI_peak_overlay_curve",
):
    bins = np.arange(0.0, 1.0 + bin_width, bin_width)
    bin_centers = (bins[:-1] + bins[1:]) / 2.0
    kernel = _gaussian_kernel1d(sigma_bins=smooth_sigma_bins)

    fig, ax = plt.subplots()

    for label, synapse_func_all in synapse_func_dict.items():
        peak_x = _extract_peak_x_numeric(
            synapse_func_all, "gDSI", bins, bin_centers,
            group_col=group_col, clip_range=(0.0, 1.0)
        )
        if peak_x.size == 0:
            continue
        y, _ = np.histogram(peak_x, bins=bins, density=density)
        ax.plot(bin_centers, np.convolve(y, kernel, mode="same"),
                alpha=alpha, linewidth=linewidth, label=f"{label} (n={peak_x.size})")

    if synapse_func_exc is not None:
        peak_x_exc = _extract_peak_x_numeric(
            synapse_func_exc, "gDSI", bins, bin_centers,
            group_col=group_col, clip_range=(0.0, 1.0)
        )
        if peak_x_exc.size > 0:
            y, _ = np.histogram(peak_x_exc, bins=bins, density=density)
            ax.plot(bin_centers, np.convolve(y, kernel, mode="same"),
                    color="black", alpha=1.0, linewidth=linewidth + 0.5, zorder=10,
                    label=f"{exc_label} (n={peak_x_exc.size})")

    ax.set_xlim(0, 1)
    ax.set_xlabel("peak gDSI")
    ax.set_ylabel("density" if density else "number of neurons per bin")
    ax.set_title("Peak gDSI distribution (smoothed curves)")
    ax.legend(frameon=False)
    plt.tight_layout()

    if save_opt:
        plt.savefig(os.path.join(save_dir, f"{fname}.png"), dpi=300)
        plt.savefig(os.path.join(save_dir, f"{fname}.svg"), dpi=300)
        plt.close(fig)
    else:
        plt.show()


In [138]:
# === OVERLAYED PLOT cc_abs peak distribution plot ===
def plot_cc_abs_peak_distribution_step(
    synapse_func_dict,
    synapse_func_exc=None,
    exc_label="Exc",
    bin_width=0.05,
    group_col="post_pt_root_id",
    density=True,
    alpha=0.8,
    linewidth=2.0,
    save_opt=True,
    save_dir="",
    fname="cc_abs_peak_overlay_step",
):
    bins = np.arange(0.0, 1.0 + bin_width, bin_width)
    bin_centers = (bins[:-1] + bins[1:]) / 2.0

    fig, ax = plt.subplots()

    for label, synapse_func_all in synapse_func_dict.items():
        peak_x = _extract_peak_x_numeric(
            synapse_func_all, "cc_abs", bins, bin_centers,
            group_col=group_col, clip_range=(0.0, 1.0)
        )
        if peak_x.size == 0:
            continue
        ax.hist(peak_x, bins=bins, density=density, histtype="step",
                linewidth=linewidth, alpha=alpha, label=f"{label} (n={peak_x.size})")

    if synapse_func_exc is not None:
        peak_x_exc = _extract_peak_x_numeric(
            synapse_func_exc, "cc_abs", bins, bin_centers,
            group_col=group_col, clip_range=(0.0, 1.0)
        )
        if peak_x_exc.size > 0:
            ax.hist(peak_x_exc, bins=bins, density=density, histtype="step",
                    linewidth=linewidth + 0.5, alpha=1.0, color="black", zorder=10,
                    label=f"{exc_label} (n={peak_x_exc.size})")

    ax.set_xlim(0, 1)
    ax.set_xlabel("peak cc_abs")
    ax.set_ylabel("density" if density else "number of neurons")
    ax.set_title("Distribution of peak cc_abs (overlay)")
    ax.legend(frameon=False)
    plt.tight_layout()

    if save_opt:
        plt.savefig(os.path.join(save_dir, f"{fname}.png"), dpi=300)
        plt.savefig(os.path.join(save_dir, f"{fname}.svg"), dpi=300)
        plt.close(fig)
    else:
        plt.show()

# === CURVED OVERLAYED ===
def plot_cc_abs_peak_distribution_curve(
    synapse_func_dict,
    synapse_func_exc=None,
    exc_label="Exc",
    bin_width=0.05,
    group_col="post_pt_root_id",
    density=True,
    smooth_sigma_bins=1.0,
    alpha=0.9,
    linewidth=2.5,
    save_opt=True,
    save_dir="",
    fname="cc_abs_peak_overlay_curve",
):
    bins = np.arange(0.0, 1.0 + bin_width, bin_width)
    bin_centers = (bins[:-1] + bins[1:]) / 2.0
    kernel = _gaussian_kernel1d(sigma_bins=smooth_sigma_bins)

    fig, ax = plt.subplots()

    for label, synapse_func_all in synapse_func_dict.items():
        peak_x = _extract_peak_x_numeric(
            synapse_func_all, "cc_abs", bins, bin_centers,
            group_col=group_col, clip_range=(0.0, 1.0)
        )
        if peak_x.size == 0:
            continue
        y, _ = np.histogram(peak_x, bins=bins, density=density)
        ax.plot(bin_centers, np.convolve(y, kernel, mode="same"),
                alpha=alpha, linewidth=linewidth, label=f"{label} (n={peak_x.size})")

    if synapse_func_exc is not None:
        peak_x_exc = _extract_peak_x_numeric(
            synapse_func_exc, "cc_abs", bins, bin_centers,
            group_col=group_col, clip_range=(0.0, 1.0)
        )
        if peak_x_exc.size > 0:
            y, _ = np.histogram(peak_x_exc, bins=bins, density=density)
            ax.plot(bin_centers, np.convolve(y, kernel, mode="same"),
                    color="black", alpha=1.0, linewidth=linewidth + 0.5, zorder=10,
                    label=f"{exc_label} (n={peak_x_exc.size})")

    ax.set_xlim(0, 1)
    ax.set_xlabel("peak cc_abs")
    ax.set_ylabel("density" if density else "number of neurons per bin")
    ax.set_title("Peak cc_abs distribution (smoothed curves)")
    ax.legend(frameon=False)
    plt.tight_layout()

    if save_opt:
        plt.savefig(os.path.join(save_dir, f"{fname}.png"), dpi=300)
        plt.savefig(os.path.join(save_dir, f"{fname}.svg"), dpi=300)
        plt.close(fig)
    else:
        plt.show()


### Plot circular variance

In [53]:
# === Circular Variance for preferred orientation === 
def circ_stats_orientation_rad(theta_rad):
    """
    theta_rad: 1D array of angles in radians (orientation, 0..pi periodic)
    Returns: (R, circ_var, circ_mean_deg_0_180)
    """
    theta = np.asarray(theta_rad, dtype=float)
    theta = theta[~np.isnan(theta)]
    if theta.size == 0:
        return np.nan, np.nan, np.nan

    # Double-angle trick for axial/orientation data
    ang = 2.0 * theta
    C = np.mean(np.cos(ang))
    S = np.mean(np.sin(ang))
    R = np.sqrt(C**2 + S**2)
    circ_var = 1.0 - R

    mean_rad = 0.5 * np.arctan2(S, C)
    mean_deg = (np.degrees(mean_rad) % 180.0)

    return R, circ_var, mean_deg


def compute_circvar_pref_ori(synapse_func_all, group_col="post_pt_root_id"):
    rows = []
    for neuron_id, df_neuron in synapse_func_all.groupby(group_col):
        vals = df_neuron["pref_ori"].dropna().to_numpy()
        R, V, mean_deg = circ_stats_orientation_rad(vals)
        rows.append({
            group_col: neuron_id,
            "n_used": int(vals.size),
            "R": R,
            "circ_var": V,
            "circ_mean_deg": mean_deg
        })
    return pd.DataFrame(rows)


In [54]:
# === Circular Variance for Direction === 

def circ_stats_direction_rad(theta_rad):
    """
    theta_rad: 1D array of angles in radians (direction, 0..2pi periodic)
    Returns: (R, circ_var, circ_mean_deg)
    """
    theta = np.asarray(theta_rad, dtype=float)
    theta = theta[~np.isnan(theta)]
    if theta.size == 0:
        return np.nan, np.nan, np.nan

    C = np.mean(np.cos(theta))
    S = np.mean(np.sin(theta))
    R = np.sqrt(C**2 + S**2)
    circ_var = 1.0 - R

    mean_rad = np.arctan2(S, C)
    mean_deg = (np.degrees(mean_rad) % 360.0)

    return R, circ_var, mean_deg


def compute_circvar_pref_dir(synapse_func_all, group_col="post_pt_root_id"):
    rows = []
    for neuron_id, df_neuron in synapse_func_all.groupby(group_col):
        vals = df_neuron["pref_dir"].dropna().to_numpy()
        R, V, mean_deg = circ_stats_direction_rad(vals)
        rows.append({
            group_col: neuron_id,
            "n_used": int(vals.size),
            "R": R,
            "circ_var": V,
            "circ_mean_deg": mean_deg
        })
    return pd.DataFrame(rows)


In [88]:
# == Plot - A single dataframe plot for excitatory neurons ===

def plot_circvar_single_step(
    df,                            # dataframe with column `circ_var`
    title="Circular variance (pref_ori)",
    bins=20,
    density=True,                  # True: normalized density, False: raw counts
    alpha=1.0,
    linewidth=2.0,
    color=None,                    # optional; if None matplotlib picks default
    save_opt=True,
    save_dir="",
    fname="circvar_pref_ori_single_step",
):
    """
    Single hollow step-histogram (square silhouette) of circular variance.
    Expects df to have column 'circ_var' with values ideally in [0, 1].
    """

    if "circ_var" not in df.columns:
        raise ValueError("df must contain a column named 'circ_var'.")

    vals = df["circ_var"].dropna().to_numpy()
    if vals.size == 0:
        print("No circ_var values found (all NaN or empty).")
        return

    fig, ax = plt.subplots()

    # Common bin edges on [0, 1]
    bin_edges = np.linspace(0, 1, bins + 1)

    # # Hollow graph
    # ax.hist(
    #     vals,
    #     bins=bin_edges,
    #     density=density,
    #     histtype="step",
    #     linewidth=linewidth,
    #     alpha=alpha,
    #     color=color,
    # )

    # Filled graph
    ax.hist(
        vals,
        bins=bin_edges,
        density=density,
        histtype="bar",      # filled histogram
        linewidth=linewidth,
        alpha=alpha,
        color="grey",
        edgecolor="black",
    )


    ax.set_xlim(0, 1)
    ax.set_xlabel("circular variance")
    ax.set_ylabel("density" if density else "number of neurons")
    ax.set_title(title)

    plt.tight_layout()

    if save_opt:
        os.makedirs(save_dir, exist_ok=True) if save_dir else None
        plt.savefig(os.path.join(save_dir, f"{fname}.png"), dpi=300, format="png")
        plt.savefig(os.path.join(save_dir, f"{fname}.svg"), dpi=300, format="svg")
        plt.close(fig)
    else:
        plt.show()


In [56]:
# == Plot - overlay with shillouette for inhibitory neurons ===

def plot_circvar_overlay_step(
    circvar_dict,                 # {"BC": df, "BPC": df, ...}
    title="Circular variance (pref_ori)",
    bins=20,
    density=True,   # True if normalized, False to show you full counts
    alpha=0.8,
    linewidth=2.0,
    save_opt=True,
    save_dir="",
    fname="circvar_pref_ori_overlay_step",
):
    """
    Overlaid hollow step-histograms (square silhouettes) of circular variance.
    Each item in circvar_dict is a dataframe with column 'circ_var'.
    """

    fig, ax = plt.subplots()

    # Common bin edges for everyone on [0, 1]
    bin_edges = np.linspace(0, 1, bins + 1)

    for label, df in circvar_dict.items():
        vals = df["circ_var"].dropna().to_numpy()
        if vals.size == 0:
            continue

        ax.hist(
            vals,
            bins=bin_edges,
            density=density,
            histtype="step",      # <-- hollow square silhouette
            linewidth=linewidth,
            alpha=alpha,
            label=label,
        )

    ax.set_xlim(0, 1)
    ax.set_xlabel("circular variance")
    ax.set_ylabel("density" if density else "number of neurons")
    ax.set_title(title)
    ax.legend(frameon=False)

    plt.tight_layout()

    if save_opt:
        plt.savefig(os.path.join(save_dir, f"{fname}.png"), dpi=300, format="png")
        plt.savefig(os.path.join(save_dir, f"{fname}.svg"), dpi=300, format="svg")
        plt.close(fig)
    else:
        plt.show()


# Main

## Inhibitory Plots

### Presynaptic input feature distributions

In [83]:
path_save = path_project + "results_inhibitory/presynaptic_dist_basic/"

In [67]:
# Plot and save presynaptic input feature distribution
synapse_func_all = synapse_func_NGC
neuron_type = "NGC"
save_opt = True

# === 1) presynaptic preferred orientation ===


pref_ori_plot(
    synapse_func_all, 
    neuron_type = neuron_type,
    bin_width_deg=15, 
    group_col='post_pt_root_id', 
    save_opt = save_opt, 
    save_dir = path_save)

# === 2) presynaptic preferred direction ===


pref_dir_plot(
    synapse_func_all,
    neuron_type = neuron_type,
    bin_width_deg=30,
    group_col="post_pt_root_id",  # per neuron
    save_opt=save_opt,
    save_dir=path_save
)

# === 3) presynaptic gOSI ===

gosi_plot(
    synapse_func_all,
    neuron_type = neuron_type,
    bin_width=0.05,
    group_col="post_pt_root_id",  # per neuron
    save_opt=save_opt,
    save_dir=path_save
)

# === 4) presynaptic gDSI ===

gdsi_plot(
    synapse_func_all,
    neuron_type = neuron_type,
    bin_width=0.05,
    group_col="post_pt_root_id",  # per neuron
    save_opt=save_opt,
    save_dir=path_save
)

# === 5) cc_abs plot ===

cc_abs_plot(
    synapse_func_all,
    neuron_type = neuron_type,
    bin_width=0.05,
    group_col="post_pt_root_id",  # per neuron
    save_opt=save_opt,
    save_dir=path_save
)

### Presynaptic input feature distributions (peaks only)

Here we only bring peak points of the distribution from above per each neuron, and then see the distribution of them

In [59]:
path_save = path_project + "results_inhibitory/presynaptic_dist_peak/"

In [137]:
# === Individual plot per cell types ===
synapse_func_all = synapse_func_NGC
neuron_type = "NGC"
save_opt = False

# === 1) peak preferred orientation ===
plot_pref_ori_peak_distribution_single(
    synapse_func_all,
    neuron_type = neuron_type,
    bin_width_deg=15,
    group_col="post_pt_root_id",
    save_opt = save_opt,  # Boolean: true to save false to see the plot
    save_dir = path_save
)

# === 2) peak preferred direction ===
plot_pref_dir_peak_distribution_single(
    synapse_func_all,
    neuron_type = neuron_type,
    bin_width_deg = 30,
    group_col = "post_pt_root_id",
    save_opt = save_opt,
    save_dir = path_save
)

# === 3) peak gOSI distribution ===
plot_gosi_peak_distribution_single(
    synapse_func_all,
    neuron_type=neuron_type,
    bin_width=0.05,
    group_col="post_pt_root_id",
    save_opt=save_opt,
    save_dir=path_save
)

# === 4) peak gDSI distribution ===
plot_gdsi_peak_distribution_single(
    synapse_func_all,
    neuron_type=neuron_type,
    bin_width=0.05,
    group_col="post_pt_root_id",
    save_opt=save_opt,
    save_dir=path_save,
)

# === 5) peak cc_abs distribution ===
plot_cc_abs_peak_distribution_single(
    synapse_func_all,
    neuron_type=neuron_type,
    bin_width=0.05,
    group_col="post_pt_root_id",
    save_opt=save_opt,
    save_dir=path_save,
)

In [61]:
# === Overlay multiple plots ===
# Plot multiple cell types together their distributions of presynaptic input feature distribution

synapse_dict = {
    "BC":  synapse_func_BC_clean,
    "BPC": synapse_func_BPC_clean,
    "MC":  synapse_func_MC_clean,
    "NGC": synapse_func_NGC_clean,
}
density = True     # True - normalized, False - actual count (original histogram)
save_opt = True
suffix = "_normalized" if density else ""


# Preferred orientation
plot_pref_ori_peak_distribution(
    synapse_dict,
    bin_width_deg=15,
    group_col="post_pt_root_id",
    density=density,    
    alpha=0.8,
    linewidth=2.0,
    save_opt=save_opt,
    save_dir=path_save,
    fname=f"pref_ori_peak_overlay_step{suffix}"
)

# Preferred direction
plot_pref_dir_peak_distribution(
    synapse_dict,
    bin_width_deg=30,
    group_col="post_pt_root_id",
    density=density,
    alpha=0.8,
    linewidth=2.0,
    save_opt=save_opt,
    save_dir=path_save,
    fname=f"pref_dir_peak_overlay_step{suffix}"
)

# GOSI
plot_gosi_peak_distribution(
    synapse_dict, 
    bin_width=0.05, 
    density=density, 
    save_opt=save_opt, 
    save_dir=path_save,
    fname=f"gOSI_peak_overlay_step{suffix}"
)

# GDSI
plot_gdsi_peak_distribution(
    synapse_dict,
    bin_width=0.05,
    density=density,
    save_opt=save_opt,
    save_dir=path_save,
    fname=f"gDSI_peak_overlay_step{suffix}"
)

# cc_abs
plot_cc_abs_peak_distribution(
    synapse_dict,
    bin_width=0.05,
    density=density,
    save_opt=save_opt,
    save_dir=path_save,
    fname=f"cc_abs_peak_overlay_step{suffix}"
)



### Presynaptic input circular variance

In [65]:
path_save = path_project + "results_inhibitory/presynaptic_circular_variance/"

In [66]:
# Calculate circular preferred orientation and plot them together

circ_BC  = compute_circvar_pref_ori(synapse_func_BC_clean)
circ_BPC = compute_circvar_pref_ori(synapse_func_BPC_clean)
circ_MC  = compute_circvar_pref_ori(synapse_func_MC_clean)
circ_NGC = compute_circvar_pref_ori(synapse_func_NGC_clean)

circ_dict = {"BC": circ_BC, "BPC": circ_BPC, "MC": circ_MC, "NGC": circ_NGC}

plot_circvar_overlay_step(
    circ_dict,
    title="Presynaptic orientation circular variance (per postsynaptic neuron)",
    bins=20,
    density=False,
    alpha=0.8,
    linewidth=2.0,
    save_opt=True,
    save_dir=path_save,
    fname="circvar_pref_ori_overlay_step_total_count"
)

In [67]:
# Calculate circular preferred direction and plot them together

circ_BC_dir  = compute_circvar_pref_dir(synapse_func_BC_clean)
circ_BPC_dir = compute_circvar_pref_dir(synapse_func_BPC_clean)
circ_MC_dir  = compute_circvar_pref_dir(synapse_func_MC_clean)
circ_NGC_dir = compute_circvar_pref_dir(synapse_func_NGC_clean)

circ_dict_dir = {"BC": circ_BC_dir, "BPC": circ_BPC_dir, "MC": circ_MC_dir, "NGC": circ_NGC_dir}

plot_circvar_overlay_step(
    circ_dict_dir,
    title="Presynaptic direction circular variance (per postsynaptic neuron)",
    bins=20,
    density=False,
    alpha=0.8,
    linewidth=2.0,
    save_opt=True,
    save_dir=path_save,
    fname="circvar_pref_dir_overlay_step_total_count"
)

## Excitatory Plots (for comparison with inhibitory neurons)

### Presynaptic input features

In [82]:
path_save = path_project + "results_excitatory/presynaptic_dist_peak/"

In [83]:
synapse_func_all = synapse_func_23P_clean
neuron_type = "23P"
save_opt = True

# === 1) peak preferred orientation ===
plot_pref_ori_peak_distribution_single(
    synapse_func_all,
    neuron_type=neuron_type,
    bin_width_deg=15,
    group_col="post_pt_root_id",
    save_opt = save_opt,  # Boolean: true to save false to see the plot
    save_dir = path_save,
)

# === 2) peak preferred direction ===
plot_pref_dir_peak_distribution_single(
    synapse_func_all,
    neuron_type=neuron_type,
    bin_width_deg=30,
    group_col="post_pt_root_id",
    save_opt=save_opt,
    save_dir=path_save,
)

# === 3) peak gOSI distribution ===
plot_gosi_peak_distribution_single(
    synapse_func_all,
    neuron_type=neuron_type,
    bin_width=0.05,
    group_col="post_pt_root_id",
    save_opt=save_opt,
    save_dir=path_save
)

# === 4) peak gDSI distribution ===
plot_gdsi_peak_distribution_single(
    synapse_func_all,
    neuron_type=neuron_type,
    bin_width=0.05,
    group_col="post_pt_root_id",
    save_opt=save_opt,
    save_dir=path_save,
)

# === 5) peak cc_abs distribution ===
plot_cc_abs_peak_distribution_single(
    synapse_func_all,
    neuron_type=neuron_type,
    bin_width=0.05,
    group_col="post_pt_root_id",
    save_opt=save_opt,
    save_dir=path_save,
)


### Presynaptic circular variance

In [104]:
path_save = path_project + "results_excitatory/presynaptic_circular_variance/"

In [105]:
# Calculate circular preferred orientation and plot them together
circ_23P = compute_circvar_pref_ori(synapse_func_23P_clean)

plot_circvar_single_step(
    circ_23P,                            # dataframe with column `circ_var`
    title="Circular variance (pref_ori)",
    bins=20,
    density=True,                  # True: normalized density, False: raw counts
    alpha=1.0,
    linewidth=0.7,
    color=None,                    # optional; if None matplotlib picks default
    save_opt=True,
    save_dir=path_save,
    fname="circvar_pref_ori_single_step",
)

In [106]:
# Calculate circular preferred direction and plot them together
circ_23P_dir = compute_circvar_pref_dir(synapse_func_23P_clean)

plot_circvar_single_step(
    circ_23P_dir,                            # dataframe with column `circ_var`
    title="Circular variance (pref_dir)",
    bins=20,
    density=True,                  # True: normalized density, False: raw counts
    alpha=1.0,
    linewidth=0.7,
    color=None,                    # optional; if None matplotlib picks default
    save_opt=True,
    save_dir=path_save,
    fname="circvar_pref_dir_single_step",
)

## Excitatory and Inhibitory Plot together

In [123]:
path_save = path_project + "results_exc_inhi_together/"

In [148]:
inh_dict = {
    "BC": synapse_func_BC_clean, 
    "BPC": synapse_func_BPC_clean, 
    "MC": synapse_func_MC_clean, 
    "NGC": synapse_func_NGC_clean}
exc_df = synapse_func_23P_clean  # or None if you don't want to include inhibitory

density = True      # True for normalized; False for raw counts
save_opt = True
smooth_sigma_bins = 1.0     # try 0.8 / 1.0 / 1.3
suffix = "_normalized" if density else ""

# Plot features both in block and curve style
plot_pref_ori_peak_distribution(synapse_dict, synapse_func_exc=synapse_func_23P_clean, exc_label="23P", bin_width_deg=15, group_col="post_pt_root_id",
    density=density, alpha=0.8, linewidth=2.0, save_opt=save_opt, save_dir=path_save, fname=f"pref_ori_peak_overlay_step_inh_plus_exc{suffix}")
plot_pref_ori_peak_distribution_curve(synapse_dict, synapse_func_exc=synapse_func_23P_clean, exc_label="23P", bin_width_deg=15, group_col="post_pt_root_id",
    density=density, smooth_sigma_bins=smooth_sigma_bins, save_opt=save_opt, save_dir=path_save, fname=f"pref_ori_peak_overlay_curve_normalized{suffix}")

plot_pref_dir_peak_distribution_step(inh_dict, synapse_func_exc=exc_df, exc_label="23P", density=density, save_opt=save_opt, save_dir=path_save)
plot_pref_dir_peak_distribution_curve(inh_dict, synapse_func_exc=exc_df, exc_label="23P", density=density, smooth_sigma_bins=1.0, save_opt=save_opt, save_dir=path_save)

plot_gosi_peak_distribution_step(inh_dict, synapse_func_exc=exc_df, exc_label="23P", density=density, save_opt=save_opt, save_dir=path_save)
plot_gosi_peak_distribution_curve(inh_dict, synapse_func_exc=exc_df, exc_label="23P", density=density, smooth_sigma_bins=1.0, save_opt=save_opt, save_dir=path_save)

plot_gdsi_peak_distribution_step(inh_dict, synapse_func_exc=exc_df, exc_label="23P", density=density, save_opt=save_opt, save_dir=path_save)
plot_gdsi_peak_distribution_curve(inh_dict, synapse_func_exc=exc_df, exc_label="23P", density=density, smooth_sigma_bins=1.0, save_opt=save_opt, save_dir=path_save)

plot_cc_abs_peak_distribution_step(inh_dict, synapse_func_exc=exc_df, exc_label="23P", density=density, save_opt=save_opt, save_dir=path_save)
plot_cc_abs_peak_distribution_curve(inh_dict, synapse_func_exc=exc_df, exc_label="23P", density=density, smooth_sigma_bins=1.0, save_opt=save_opt, save_dir=path_save)